# 02 - Train and Evaluate YOLOv9-C
This notebook trains YOLOv9-C on the vehicle dataset using pretrained weights, evaluates performance on the test set, and displays sample visual predictions.

In [ ]:
# 1. Imports and environment check
import os
import sys
import urllib.request
from pathlib import Path
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU device:     ", torch.cuda.get_device_name(0))
else:
    print("Warning: CUDA is not available. Please connect to GPU on Vast.ai.")

In [ ]:
# 2. Install dependencies
# Installs requirements into the current Python environment
!pip install -q -r ../yolov9/requirements.txt

In [ ]:
# 3. Setup paths and working directory
project_dir = Path.cwd().parent
yolo_dir = project_dir / "yolov9"
data_yaml = project_dir / "roadwatcher.yaml"
data_dir = project_dir / "data_clean"

print("Project folder: ", project_dir)
print("YOLO directory: ", yolo_dir)
print("Data YAML:      ", data_yaml)

assert data_yaml.exists(), "roadwatcher.yaml not found! Run 01_prepare_data.ipynb first."

# Change working directory to yolov9 so scripts run cleanly
os.chdir(yolo_dir)
print("Current directory:", Path.cwd())

In [ ]:
# 4. Download pretrained weights (yolov9-c.pt)
weights_file = Path("yolov9-c.pt")

if not weights_file.exists():
    print("Downloading yolov9-c.pt pretrained weights...")
    url = "https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt"
    urllib.request.urlretrieve(url, "yolov9-c.pt")
    print("Download finished!")
else:
    print("Pretrained weights already exist:", weights_file.resolve())

## Train YOLOv9-C
Fixed brief settings:
- Architecture: `models/detect/yolov9-c.yaml`
- Weights: `yolov9-c.pt`
- Image size: 640
- Batch size: 4
- Epochs: 10
- Hyperparameters: `data/hyps/hyp.scratch-high.yaml`

In [ ]:
# 5. Launch training with train_dual.py
!python train_dual.py \
    --device 0 \
    --workers 2 \
    --img 640 \
    --batch-size 4 \
    --epochs 10 \
    --data ../roadwatcher.yaml \
    --cfg models/detect/yolov9-c.yaml \
    --weights yolov9-c.pt \
    --hyp data/hyps/hyp.scratch-high.yaml \
    --name roadwatcher_yolov9c \
    --min-items 0 \
    --close-mosaic 2 \
    --exist-ok

In [ ]:
# 6. Check checkpoints and display training loss curves
from IPython.display import Image as DisplayImage, display

best_pt = Path("runs/train/roadwatcher_yolov9c/weights/best.pt")
last_pt = Path("runs/train/roadwatcher_yolov9c/weights/last.pt")

print("best.pt exists:", best_pt.exists())
print("last.pt exists:", last_pt.exists())

results_png = Path("runs/train/roadwatcher_yolov9c/results.png")
if results_png.exists():
    display(DisplayImage(filename=str(results_png)))
else:
    print("results.png not found yet (will be created after training completes).")

## Evaluate Model on Test Set
Evaluate the trained `best.pt` model on the 405 test images.
Note: We use `--conf-thres 0.001` for evaluation so mAP considers all possible detections.

In [ ]:
# 7. Evaluate on the 405 test images
!python val_dual.py \
    --data ../roadwatcher.yaml \
    --weights runs/train/roadwatcher_yolov9c/weights/best.pt \
    --img 640 \
    --batch-size 4 \
    --conf-thres 0.001 \
    --iou-thres 0.7 \
    --task test \
    --verbose \
    --exist-ok

## Visual Predictions on Test Set
Predict on test images using `--conf-thres 0.25` for clean, human-readable bounding boxes.

In [ ]:
# 8. Run inference on test images
!python detect_dual.py \
    --weights runs/train/roadwatcher_yolov9c/weights/best.pt \
    --source ../data_clean/test/images \
    --data ../roadwatcher.yaml \
    --img 640 \
    --conf-thres 0.25 \
    --iou-thres 0.45 \
    --project runs/detect \
    --name roadwatcher_test \
    --exist-ok

In [ ]:
# 9. Display sample predictions
pred_dir = Path("runs/detect/roadwatcher_test")
pred_images = sorted(list(pred_dir.glob("*.jpg")) + list(pred_dir.glob("*.png")))

print("Total prediction images generated:", len(pred_images))
print("Displaying first 6 predictions:")

for img_path in pred_images[:6]:
    print(img_path.name)
    display(DisplayImage(filename=str(img_path), width=500))

## Summary of Important Outputs to Download
Before stopping or deleting your Vast.ai GPU instance, make sure to download:
1. `yolov9/runs/train/roadwatcher_yolov9c/weights/best.pt`
2. `yolov9/runs/train/roadwatcher_yolov9c/weights/last.pt`
3. `yolov9/runs/train/roadwatcher_yolov9c/results.png`
4. Sample detection images from `yolov9/runs/detect/roadwatcher_test/`